# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```
The FAIR^2 dataset includes ordered logistic regression outputs and socio-demographic predictors for rangeland management adoption, described by a full Croissant schema.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using the mlcroissant Dataset class
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")
print(f"Data biases: {getattr(metadata, 'dataBiases', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined in the Croissant schema.<br>
We'll list all registered Record Sets and their corresponding `@id`s and fields. Each Record Set represents a table or key entity in the dataset.

In [ ]:
# Explore the Record Sets present in metadata
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record set(s) in the dataset.\n")
for rs in record_sets:
    print(f"Record Set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - {fld.name} (@id: {fld.id}, type: {getattr(fld, 'data_type', 'unknown')})")
    print()

### Record Sample
Let's take a look at a sample record from each Record Set using their `@id`. This helps us understand the structure and field coverage.

In [ ]:
# Print a sample record from each record set by `@id`.
for rs in record_sets:
    print(f"Sample record for Record Set: {rs.name} (@id: {rs.id})")
    sample_record = next(dataset.records(record_set=rs.id), None)
    if sample_record is not None:
        print(json.dumps(sample_record, indent=2))
    else:
        print("  No records found in this record set.")
    print('-' * 60)

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for further analysis.<br>
All references—including record sets and fields—use the `@id` for reproducibility and cross-tool compatibility.

In [ ]:
# Extract data from all record sets found above, by their `@id`.
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display the columns of the first record set as an example
if record_set_ids:
    select_id = record_set_ids[0]
    print(f"Columns in record set '@id': {select_id}")
    print(dataframes[select_id].columns.tolist())
    dataframes[select_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate typical EDA steps on a numeric field within a record set. 
You can adjust the field `@id` and record set `@id` to experiment with different attributes.

In [ ]:
# Select record set and numeric field by their `@id` (customize as needed)
example_record_set_id = select_id
df = dataframes[example_record_set_id]
print(f"DataFrame shape for {example_record_set_id}: {df.shape}")

# Attempt automatic numeric field detection (replace with a specific id if needed)
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' and not col.startswith('Unnamed')]
numeric_field_id = numeric_candidates[0] if numeric_candidates else None

if numeric_field_id:
    print(f"Using numeric field '@id': {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    threshold = float(threshold)

    # Filter records above threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean): {len(filtered_df)} found\n")

    # Normalize
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Top 5 normalized entries for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try grouping (pick first non-numeric column as group field)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and not df[col].dtype.kind in 'fi':
            group_field_id = col
            break

    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
        print(grouped.head())
else:
    print("No suitable numeric field found for EDA. Column types:")
    print(df.dtypes)

## 5. Visualization
Visualize the distribution of the selected numeric field and any interesting relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset as defined by its Croissant schema using the `mlcroissant` library.

- The dataset provides rich logistic regression outputs and socio-demographic survey data relevant to rangeland management in Northern Kenya.
- We enumerated Record Sets and their `@id`s, loaded data into DataFrames by `@id`, and demonstrated a simple numeric EDA/visualization pipeline.
- You are encouraged to inspect additional record sets, dig into categorical or text-based attributes, and experiment with customized visualizations as needed for your research or applications.

For more on the schema and data lineage, see the official [FAIR^2 record](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

Happy data science!